# YOLO11 Fire Detection — Google Colab (Google Drive)

**Model:** `leeyunjai/yolo11-firedetect` — YOLO11s fine-tuned for fire & smoke detection.

**Output per image (saved to `yolo11_fire_detection.csv`):**

| filename | fire_detected | num_detections | labels | max_confidence |
|---|---|---|---|---|
- `fire_detected` — `True` if at least one detection above the confidence threshold
- `num_detections` — total number of bounding boxes detected
- `labels` — semicolon-separated list of detected class names
- `max_confidence` — highest confidence score across all detections (0.0 if none)

### Before running:

1. Go to [drive.google.com](https://drive.google.com) and upload your image folder into **My Drive**
2. Set `DRIVE_IMAGE_FOLDER` in Cell 1 to the exact folder name you uploaded
3. **Allow popups** for `colab.research.google.com` — the Drive mount opens a Google sign-in popup

**Runtime recommendation:** `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ── Cell 1: Mount Google Drive and set image folder ───────────────────────────
from google.colab import drive
import os, zipfile

drive.mount('/content/drive', force_remount=True)

# ↓ Change this to the name of the folder you uploaded to My Drive
DRIVE_IMAGE_FOLDER = 'yolo11images'

DRIVE_DIR = f'/content/drive/MyDrive/{DRIVE_IMAGE_FOLDER}'

assert os.path.isdir(DRIVE_DIR), (
    f'Folder not found: {DRIVE_DIR}\n'
    f'Make sure "{DRIVE_IMAGE_FOLDER}" exists inside My Drive on Google Drive.'
)

# Extract ZIPs if present
zip_files = [f for f in os.listdir(DRIVE_DIR) if f.lower().endswith('.zip')]
if zip_files:
    EXTRACT_DIR = f'/content/{DRIVE_IMAGE_FOLDER}_extracted'
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    for zf in zip_files:
        print(f'Extracting {zf} ...')
        with zipfile.ZipFile(os.path.join(DRIVE_DIR, zf), 'r') as z:
            z.extractall(EXTRACT_DIR)
    IMAGE_DIR = EXTRACT_DIR
    print(f'Extracted to {IMAGE_DIR}')
else:
    IMAGE_DIR = DRIVE_DIR
    print(f'Reading images directly from {IMAGE_DIR}')

total = sum(1 for _, _, fs in os.walk(IMAGE_DIR) for f in fs)
print(f'Found {total} file(s) in total')

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
!pip install -q ultralytics
print('Done.')

In [ ]:
# ── Cell 3: Download YOLO11 fire-detection weights ────────────────────────────
import torch
from ultralytics import YOLO

WEIGHTS_PATH = '/content/fire_model.pt'
WEIGHTS_URL  = 'https://huggingface.co/leeyunjai/yolo11-firedetect/resolve/main/firedetect-11s.pt'

if not os.path.exists(WEIGHTS_PATH):
    print('Downloading weights ...')
    os.system(f'wget -q -O {WEIGHTS_PATH} {WEIGHTS_URL}')
else:
    print('Weights already downloaded.')

model = YOLO(WEIGHTS_PATH)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Fire Detection Model loaded  |  device: {DEVICE.upper()}')
print(f'Classes: {model.names}')

In [ ]:
# ── Cell 4: Configuration ─────────────────────────────────────────────────────
CONF_THRESHOLD = 0.25   # minimum confidence to count a detection
IMAGE_EXTS     = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}

# Output CSV path (saved to Drive so it persists after the session ends)
OUTPUT_CSV = f'/content/drive/MyDrive/yolo11_fire_detection.csv'

In [ ]:
# ── Cell 5: Run detection on all images ───────────────────────────────────────
import csv
import time
from pathlib import Path

# Collect all image paths recursively
image_paths = sorted(
    p for p in Path(IMAGE_DIR).rglob('*')
    if p.suffix.lower() in IMAGE_EXTS
)
print(f'Found {len(image_paths)} image(s). Starting detection ...')

rows = []
t0   = time.time()

for i, img_path in enumerate(image_paths, 1):
    results = model.predict(
        source=str(img_path),
        conf=CONF_THRESHOLD,
        device=DEVICE,
        verbose=False,
        save=False,
    )

    result = results[0]
    boxes  = result.boxes

    if boxes is not None and len(boxes) > 0:
        confs       = boxes.conf.cpu().tolist()
        class_ids   = boxes.cls.cpu().int().tolist()
        label_names = [model.names[c] for c in class_ids]
        num_det     = len(boxes)
        max_conf    = max(confs)
        labels_str  = ';'.join(label_names)
        fire_det    = True
    else:
        num_det    = 0
        max_conf   = 0.0
        labels_str = ''
        fire_det   = False

    rows.append({
        'filename':       img_path.name,
        'fire_detected':  fire_det,
        'num_detections': num_det,
        'labels':         labels_str,
        'max_confidence': round(max_conf, 4),
    })

    if i % 50 == 0 or i == len(image_paths):
        elapsed = time.time() - t0
        print(f'  [{i}/{len(image_paths)}]  elapsed: {elapsed:.1f}s')

# Save CSV
fieldnames = ['filename', 'fire_detected', 'num_detections', 'labels', 'max_confidence']
with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

total_time = time.time() - t0
detected   = sum(1 for r in rows if r['fire_detected'])
print(f'\nDone in {total_time:.1f}s')
print(f'Fire detected in {detected}/{len(rows)} images')
print(f'Results saved to: {OUTPUT_CSV}')

In [ ]:
# ── Cell 6: Preview results ────────────────────────────────────────────────────
import pandas as pd

df = pd.read_csv(OUTPUT_CSV)
print(f'Total images : {len(df)}')
print(f'Fire detected: {df["fire_detected"].sum()}  ({df["fire_detected"].mean()*100:.1f}%)')
print()
print('── First 10 rows ──')
display(df.head(10))

In [ ]:
# ── Cell 7 (optional): Visualise detections on sample images ──────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image as PILImage

# Show up to N images where fire was detected
N_SHOW = 6

detected_paths = [
    p for p in image_paths
    if df.loc[df['filename'] == p.name, 'fire_detected'].values[0]
][:N_SHOW]

if not detected_paths:
    print('No fire detections to visualise.')
else:
    fig, axes = plt.subplots(1, len(detected_paths), figsize=(5 * len(detected_paths), 5))
    if len(detected_paths) == 1:
        axes = [axes]

    for ax, img_path in zip(axes, detected_paths):
        res = model.predict(
            source=str(img_path),
            conf=CONF_THRESHOLD,
            device=DEVICE,
            verbose=False,
            save=False,
        )[0]
        ax.imshow(res.plot()[:, :, ::-1])   # BGR → RGB
        ax.set_title(img_path.name, fontsize=8)
        ax.axis('off')

    plt.suptitle('YOLO11 Fire Detection — Sample Results', fontsize=12)
    plt.tight_layout()
    plt.show()